<font size="+3"><strong>5.5. Bankruptcy in Taiwan ????</strong></font>

In [ ]:
# Import libraries here
import gzip
import json
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from IPython.display import VimeoVideo
from ipywidgets import interact
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from teaching_tools.widgets import ConfusionMatrixWidget

# Prepare Data

## Import

**Task 5.5.1**

In [ ]:
# Load data file
with gzip.open("data/taiwan-bankruptcy-data.json.gz") as rf:
    taiwan_data = json.load(rf)
print(type(taiwan_data))

**Task 5.5.2**

<div class="alert alert-info" role="alert">
    <b>Tip:</b> The data in this assignment might be organized differently than the data from the project, so be sure to inspect it first. 
</div>

In [ ]:
taiwan_data_keys = taiwan_data.keys()
print(taiwan_data_keys)

**Task 5.5.3**

In [ ]:
n_companies = len(taiwan_data["observations"])
print(n_companies)

**Task 5.5.4**

In [ ]:
n_features = len(taiwan_data["observations"][67])
print(n_features)

**Task 5.5.5**

In [ ]:
# Create wrangle function
def wrangle(filepath):
    with gzip.open(filepath) as rf:
        taiwan_data = json.load(rf)

    df = pd.DataFrame().from_dict(taiwan_data["observations"]).set_index("id")
    
    return df

In [ ]:
df = wrangle("data/taiwan-bankruptcy-data.json.gz")
print("df shape:", df.shape)
df.head()

## Explore

**Task 5.5.6**

In [ ]:
nans_by_col = pd.Series(df.isna().sum())
print("nans_by_col shape:", nans_by_col.shape)
nans_by_col.head()

<div class="alert alert-info" role="alert">
  <strong>Slight Code Change</strong>

In the following task, you'll notice a small change in how plots are created compared to what you saw in the lessons.
While the lessons use the global matplotlib method like <code>plt.plot(...)</code>, in this task, you are expected to use the object-oriented (OOP) API instead.
This means creating your plots using <code>fig, ax = plt.subplots()</code> and then calling plotting methods on the <code>ax</code> object, such as <code>ax.plot(...)</code>, <code>ax.hist(...)</code>, or <code>ax.scatter(...)</code>.

If you're using pandas? or seaborn?s built-in plotting methods (like <code>df.plot()</code> or <code>sns.lineplot()</code>), make sure to pass the <code>ax=ax</code> argument so that the plot is rendered on the correct axes.

This approach is considered best practice and will be used consistently across all graded tasks that involve matplotlib.
</div>


**Task 5.5.7**

In [ ]:
# Plot class balance
fig, ax = plt.subplots()

df["bankrupt"].value_counts(normalize = True).plot(kind = "bar", ax =ax )

ax.set_xlabel("Bankrupt")
ax.set_ylabel("Frequency")
ax.set_title("Class Balance");

## Split

**Task 5.5.8**

In [ ]:
target = "bankrupt"
X = df.drop(columns = [target])
y = df[target]
print("X shape:", X.shape)
print("y shape:", y.shape)

**Task 5.5.9**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## Resample

**Task 5.5.10**

In [ ]:
over_sampler = RandomOverSampler(random_state = 42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print("X_train_over shape:", X_train_over.shape)
X_train_over.head()

# Build Model

## Iterate

**Task 5.5.11**

In [ ]:
clf = RandomForestClassifier(random_state= 42)

**Task 5.5.12**

<div class="alert alert-info" role="alert">
    <p><b>Tip:</b> Use your CV scores to evaluate different classifiers. Choose the one that gives you the best scores.</p>
</div>

In [ ]:
cv_scores = cross_val_score(clf, X_train_over, y_train_over, cv = 5, n_jobs = -1)
print(cv_scores)

In [ ]:
# Run this code cell

params = {
    "max_depth": range(30, 50, 10),
    "n_estimators": range(25, 51, 25),
}

**Task 5.5.13**

In [ ]:
model = GridSearchCV(
    clf,
    param_grid=params,
    cv = 5,
    n_jobs = -1,
    verbose = 1
)

**Ungraded Task:** Fit your model to the over-sampled training data. 

In [ ]:
model.fit(X_train_over, y_train_over)

**Task 5.5.14**

In [ ]:
cv_results = pd.DataFrame(model.cv_results_)
cv_results.head(5)

**Task 5.5.15**

In [ ]:
best_params = model.best_params_
print(best_params)

## Evaluate

**Ungraded Task:** Test the quality of your model by calculating accuracy scores for the training and test data.

In [ ]:
acc_train = model.best_estimator_.score(X_train , y_train)
acc_test = model.best_estimator_.score(X_test , y_test)

print("Model Training Accuracy:", round(acc_train, 4))
print("Model Test Accuracy:", round(acc_test, 4))

**Task 5.5.16**

In [ ]:
# write code here
fig, ax = plt.subplots()
ConfusionMatrixDisplay.from_estimator(model.best_estimator_, X_test, y_test, ax = ax);

**Task 5.5.17**

In [ ]:
class_report = classification_report(y_test, model.best_estimator_.predict(X_test))
print(class_report)

# Communicate

**Task 5.5.18**

In [ ]:
# write your code here
fig, ax = plt.subplots()
features = X_train.columns
importances =  model.best_estimator_[1].feature_importances_
feat_imp = pd.Series(importances, index = features).sort_values()

feat_imp.tail(10).plot(kind = "barh", ax=ax)
ax.set_xlabel("Gini Importance")
ax.set_ylabel("Feature")
ax.set_title("Feature Importance");

**Task 5.5.19**

In [ ]:
# Save model
with open("model-5-5.pkl", "wb") as dump:
    pickle.dump(model, dump)


**Task 5.5.20**

In [ ]:
import importlib
import my_predictor_assignment
importlib.reload(my_predictor_assignment)

In [ ]:
# Import your module
from my_predictor_assignment import make_predictions

# Generate predictions
y_test_pred = make_predictions(
    data_filepath="data/taiwan-bankruptcy-data-test-features.json.gz",
    model_filepath="model-5-5.pkl",
)

print("predictions shape:", y_test_pred.shape)
y_test_pred.head()

<div class="alert alert-info" role="alert">
    <b>Tip:</b> If you get an <code style="color:#E45E5C;background-color:#FEDDDE">ImportError</code> when you try to import <code>make_predictions</code> from <code>my_predictor_assignment</code>, try restarting your kernel. Go to the <b>Kernel</b> menu and click on  <b>Restart Kernel and Clear All Outputs</b>. Then rerun just the cell above. ??
</div>

---
Copyright 2023 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
